In [ ]:
# 1. Gỡ bản protobuf hiện tại đang gây lỗi
# !pip uninstall -y protobuf

# 2. Cài đặt bản ổn định nhất (3.20.x) tương thích với diffusers/transformers
# !pip install protobuf==3.20.3 

# 3. Cài lại các thư viện cần thiết để đảm bảo version khớp nhau
!pip install --upgrade diffusers transformers accelerate
!pip install --upgrade peft
!pip install thop

In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from torchmetrics.classification import MulticlassJaccardIndex
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torchvision.models as models

# ==========================================
# 1. CONFIGURATION
# ==========================================
# Paths
VAE_INPUT_PATH = "/kaggle/input/vae-07/pytorch/default/1/custom_seg_vae_best (1).pth" 
VAE_SAVE_PATH = "custom_seg_vae_dpx.pth" 
VAE_BEST_PATH = "custom_seg_vae_best.pth"

# Flow paths
FLOW_CHECKPOINT = "latent_flow_concat_opt2.pth"
FLOW_BEST_PATH = "best_latent_flow_concat_opt2.pth"

# Data
IMAGE_SIZE = 256
BATCH_SIZE = 32
NUM_CLASSES = 4           
COND_CHANNELS = 3         

# Model Architecture
VAE_LATENT_CHANNELS = 4   
COND_LATENT_CHANNELS = 96 

# Rectified Flow Settings
NUM_INFERENCE_STEPS = 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 2. DATASET
# ==========================================
class SemanticSegmentationDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform
        self.image_paths = sorted([os.path.join(image_dir, img) for img in os.listdir(image_dir)])
        self.label_paths = sorted([os.path.join(label_dir, lbl) for lbl in os.listdir(label_dir)])
        self.class_colors = {(255, 255, 255): 0, (160, 160, 160): 1, (80, 80, 80): 2, (0, 0, 0): 3}
     
    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        image = cv2.imread(self.image_paths[idx]); image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = cv2.imread(self.label_paths[idx]); label = cv2.cvtColor(label, cv2.COLOR_BGR2RGB)
        
        label_mask = np.zeros(label.shape[:2], dtype=np.uint8)
        for rgb, idx in self.class_colors.items():
            label_mask[np.all(label == rgb, axis=-1)] = idx
            
        if self.transform:
            image = self.transform(image)
            label_mask = torch.from_numpy(label_mask).long()
            label_onehot = F.one_hot(label_mask, num_classes=NUM_CLASSES).permute(2, 0, 1).float()
            
        return image, label_onehot

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]) 
])

# [LƯU Ý] Check lại đường dẫn của bạn
train_dataset = SemanticSegmentationDataset(
    image_dir='/kaggle/input/5g-lte-nr-j03/J03_spectrumm/train/data', 
    label_dir='/kaggle/input/5g-lte-nr-j03/J03_spectrumm/train/label', 
    transform=train_transform
)
val_dataset = SemanticSegmentationDataset(
    image_dir='/kaggle/input/5g-lte-nr-j03/J03_spectrumm/test/data', 
    label_dir='/kaggle/input/5g-lte-nr-j03/J03_spectrumm/test/label', 
    transform=train_transform
)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)


# ==============================================================================
# 1. HELPER BLOCKS & NORMALIZATION (FIXED & OPTIMIZED)
# ==============================================================================

class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

class AdaBatchNorm(nn.Module):
    def __init__(self, channels, time_dim):
        super().__init__()
        self.bn = nn.BatchNorm2d(channels, affine=False)
        self.emb = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_dim, channels * 2)
        )

    def forward(self, x, t_emb):
        scale_shift = self.emb(t_emb)[:, :, None, None]
        scale, shift = scale_shift.chunk(2, dim=1)
        x = self.bn(x)
        return x * (1 + scale) + shift


class ConvNeXtBlockBN(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, 7, padding=3, groups=dim)
        self.bn = nn.BatchNorm2d(dim)
        self.pwconv1 = nn.Conv2d(dim, 4 * dim, 1)
        self.act = nn.SiLU()
        self.pwconv2 = nn.Conv2d(4 * dim, dim, 1)
        self.bn2 = nn.BatchNorm2d(4 * dim)
    def forward(self, x):
        identity = x
        x = self.dwconv(x)
        x = self.bn(x)
        x = self.pwconv1(x)
        x = self.bn2(x)
        x = self.pwconv2(x)
        x = self.act(x)
        return x + identity


# ==============================================================================
# 2. VAE BLOCKS (GIỮ NGUYÊN LOGIC, CHỈ FIX LỖI RESUPBLOCK)
# ==============================================================================
# --- BLOCKS CHO VAE ---
class ResDownBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, 2, 1), nn.BatchNorm2d(out_c), nn.GELU(),
            nn.Conv2d(out_c, out_c, 3, 1, 1), nn.BatchNorm2d(out_c), nn.GELU()
        )
        self.skip = nn.Sequential(nn.Conv2d(in_c, out_c, 1, 2), nn.BatchNorm2d(out_c))
    def forward(self, x): return self.conv(x) + self.skip(x)

class ResUpBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_c, in_c, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, 1, 1), nn.BatchNorm2d(out_c), nn.GELU(),
            nn.Conv2d(out_c, out_c, 3, 1, 1), nn.BatchNorm2d(out_c), nn.GELU()
        )
        self.skip = nn.Conv2d(in_c, out_c, 1)
    def forward(self, x): 
        x_up = self.up(x)
        return self.conv(x_up) + self.skip(x_up)

class SegmentationVAE(nn.Module):
    def __init__(self, num_classes=4, latent_dim=4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(num_classes, 32, 3, 1, 1), nn.GELU(),
            ResDownBlock(32, 32), ResDownBlock(32, 48), ResDownBlock(48, 96),
        )
        self.fc_mu = nn.Conv2d(96, latent_dim, 1)
        self.fc_logvar = nn.Conv2d(96, latent_dim, 1)
        
        self.decoder_input = nn.Conv2d(latent_dim, 96, 1)
        self.decoder = nn.Sequential(
            ResUpBlock(96, 48), ResUpBlock(48, 32), ResUpBlock(32, 32),
            nn.Conv2d(32, num_classes, 1)
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(self.decoder_input(z))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar
# ==============================================================================
# 3. ATTENTION & DSSL BLOCKS (OPTIMIZED)
# ==============================================================================
class GatedFeatureAggregation(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate_proj = nn.Sequential(
            nn.Conv2d(dim * 3, dim, 3,1,1, bias=False),
            nn.BatchNorm2d(dim),
            nn.SiLU(), 
            nn.Conv2d(dim, dim * 3, 3,1,1, bias=False),
            nn.BatchNorm2d(dim * 3),
            nn.Sigmoid()
        )
        # self.out_proj = nn.Sequential(
        #     nn.Conv2d(dim, dim, 3, 1, 1, groups=dim, bias=False),
        #     nn.GroupNorm(8, dim),
        #     nn.SiLU() 
        # )
    def forward(self, f1, f2, f3):
        cat_feats = torch.cat([f1, f2, f3], dim=1)
        weights = self.gate_proj(cat_feats)
        w1, w2, w3 = weights.chunk(3, dim=1)
        
        aggregated = (f1 * w1) + (f2 * w2) + (f3 * w3)
        return aggregated

def haar_wavelet_decompose(x):
    B, C, H, W = x.shape
    if H % 2 != 0 or W % 2 != 0: x = F.pad(x, (0, W % 2, 0, H % 2), mode='reflect')
    x00 = x[:, :, 0::2, 0::2]; x01 = x[:, :, 0::2, 1::2]
    x10 = x[:, :, 1::2, 0::2]; x11 = x[:, :, 1::2, 1::2]
    LL = (x00 + x01 + x10 + x11) / 4
    LH = (x00 + x01 - x10 - x11) / 4
    HL = (x00 - x01 + x10 - x11) / 4
    HH = (x00 - x01 - x10 + x11) / 4
    return LL, torch.cat([LH, HL, HH], dim=1)

class DSSL(nn.Module):
    """
    Dilated Scale-Sensitive Layer (Optimized):
    Xử lý trực tiếp trên High/Low freq maps, bỏ qua bước interpolate dư thừa.
    """
    def __init__(self, channels=96, kernel_size=3, dilations=[2, 4, 8]):
        super().__init__()
        self.num_dilated_paths = len(dilations)
        self.conv_paths = nn.ModuleList()
        for d in dilations:
            self.conv_paths.append(nn.Sequential(
                nn.Conv2d(channels, channels, kernel_size, padding=d, dilation=d, groups=channels, bias=False),
                nn.BatchNorm2d(channels),
                nn.SiLU(), 
                nn.Conv2d(channels, channels, 1, bias=False)
            ))
        # Low freq processor
        self.process_LL = nn.Sequential(
            nn.Conv2d(channels, channels, 3, 1, 1, bias=False), 
            nn.BatchNorm2d(channels), nn.SiLU() 
        )
        # High freq processor
        self.process_High = nn.Sequential(
            nn.Conv2d(channels * 3, channels, 3, 1, 1, bias=False), 
            nn.BatchNorm2d(channels), nn.SiLU() 
        )
        self.global_pool = nn.AdaptiveAvgPool2d(1) 
        
        self.attention_mlp = nn.Sequential(
            nn.Conv2d(channels * 2, channels // 2, 1), 
            nn.SiLU(), 
            nn.Conv2d(channels // 2, self.num_dilated_paths * channels, 1) 
        )
        self.sigmoid = nn.Sigmoid()
        self.final_proj = nn.Conv2d(channels, channels, 1)

    def forward(self, x, cond):
        B, C, H, W = x.shape
        dilated_outs = torch.stack([conv(x) for conv in self.conv_paths], dim=1)
        
        # Haar Decompose -> Ra size H/2, W/2
        LL, High_Freqs = haar_wavelet_decompose(cond)
        
        feat_LL = self.process_LL(LL)           
        feat_High = self.process_High(High_Freqs) 
        
        # Pool trực tiếp từ size nhỏ
        pool_LL = self.global_pool(feat_LL)     
        pool_High = self.global_pool(feat_High) 
        
        ctx = torch.cat([pool_LL, pool_High], dim=1) 
        gates = self.attention_mlp(ctx)              
        gates = gates.view(B, self.num_dilated_paths, C, 1, 1) 
        gates = self.sigmoid(gates)
        
        out = torch.sum(dilated_outs * gates, dim=1) 
        return self.final_proj(out) + x


# ==========================================
# 2. CLASS ENCODER ĐÃ FIX LỖI
# ==========================================
class ResNetConditionEncoder(nn.Module):
    def __init__(self, in_channels=3, out_channels=96, base_dim=96):
        super().__init__()
        print(f"⚡ Initializing SpectraNeXtEncoder (Modern ConvNeXt-style) | Out: {out_channels}")

        # 1. Stem: Nén ảnh nhanh (256x256 -> 32x32) qua 3 bước stride=2
        self.stem = nn.Sequential(
            # Bước 1: 256 -> 128
            nn.Conv2d(in_channels, base_dim//2, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_dim//2),
            nn.SiLU(),
            nn.Conv2d(base_dim//2, base_dim//2, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(base_dim//2),
            nn.SiLU(),

            # Bước 2: 128 -> 64
            nn.Conv2d(base_dim//2, base_dim, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_dim),
            nn.SiLU(),
            nn.Conv2d(base_dim, base_dim, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(base_dim),
            nn.SiLU(),

            # Bước 3: 64 -> 32
            nn.Conv2d(base_dim, int(base_dim*2), kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(int(base_dim*2)),
            nn.SiLU(),
            nn.Conv2d(int(base_dim*2), int(base_dim*2), kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(int(base_dim*2)),
            nn.SiLU(),
        )
        
        self.final_proj = nn.Sequential(
            nn.Conv2d(int(base_dim*2), out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU()
        )

    def forward(self, x):
        x = self.stem(x)
        return self.final_proj(x)        
        
class myModel(nn.Module):
    def __init__(self, latent_channels=4, cond_channels=3, cond_inner_dim=96, out_channels=4, base_channels=96):
        super().__init__()
        print(f"🚀 Initializing Optimized UNet (Concat Only) | Base: {base_channels}")
        
        self.base_channels = base_channels
        self.cond_inner_dim = cond_inner_dim
        time_dim = base_channels * 4
        
        # 1. Condition Encoder (ResNet - Đã fix logic như bài trước)
        self.cond_enc = ResNetConditionEncoder(out_channels=cond_inner_dim)
        
        # 2. Time Embedding
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(base_channels),
            nn.Linear(base_channels, time_dim), nn.SiLU(), nn.Linear(time_dim, time_dim),
        )
        
        # 3. Stem
        self.stem = nn.Sequential(
            nn.Conv2d(latent_channels, base_channels, 3, 1, 1),
            ConvNeXtBlockBN(base_channels), ConvNeXtBlockBN(base_channels)
        )
        self.ada_stem = AdaBatchNorm(base_channels, time_dim)
        
        # STAGE 1
        self.s1_dssl1 = DSSL(base_channels, dilations=[2, 4, 8])
        self.s1_ada1  = AdaBatchNorm(base_channels, time_dim)
        self.s1_ConvNeXt = nn.Sequential(
            ConvNeXtBlockBN(base_channels + cond_inner_dim), ConvNeXtBlockBN(base_channels + cond_inner_dim)
        )
        self.s1_ada2  = AdaBatchNorm(base_channels + cond_inner_dim, time_dim)
        self.s1_proj  = nn.Conv2d(base_channels + cond_inner_dim, base_channels, 3, 1, 1)
        
        # STAGE 2
        self.s2_dssl1 = DSSL(base_channels, dilations=[2, 4, 8])
        self.s2_ada1  = AdaBatchNorm(base_channels, time_dim)
        self.s2_ConvNeXt = nn.Sequential(
            ConvNeXtBlockBN(base_channels + cond_inner_dim), ConvNeXtBlockBN(base_channels + cond_inner_dim)
        )
        self.s2_ada2  = AdaBatchNorm(base_channels + cond_inner_dim, time_dim)
        self.s2_proj  = nn.Conv2d(base_channels + cond_inner_dim, base_channels, 3, 1, 1) 
        
        # STAGE 3 
        self.s3_dssl1 = DSSL(base_channels, dilations=[2, 4, 8])
        self.s3_ada1  = AdaBatchNorm(base_channels, time_dim)
        self.s3_ConvNeXt = nn.Sequential(
            ConvNeXtBlockBN(base_channels + cond_inner_dim), ConvNeXtBlockBN(base_channels + cond_inner_dim)
        )
        self.s3_ada2  = AdaBatchNorm(base_channels + cond_inner_dim, time_dim)
        self.s3_proj = nn.Conv2d(base_channels + cond_inner_dim, base_channels, 3, 1, 1)

        # FINAL
        self.final_gated_fusion = GatedFeatureAggregation(base_channels)
        self.final_conv = nn.Conv2d(base_channels, out_channels, 3, 1, 1)

    def encode_condition(self, condition_img):
        return self.cond_enc(condition_img)

    def forward_with_cond(self, x, timesteps, cond_feats):
        t_emb = self.time_mlp(timesteps)
        x_stem = self.stem(x)
        x_stem = self.ada_stem(x_stem, t_emb)
        
        # stage 1
        x_s1 = self.s1_dssl1(x_stem,cond_feats)
        x_s1 = self.s1_ada1(x_s1, t_emb)
        
        x_in_s1 = torch.cat([x_stem, cond_feats], dim=1)  
        x_s1_res = self.s1_ConvNeXt(x_in_s1)
        feat_s1 = self.s1_proj(self.s1_ada2(x_s1_res, t_emb))
        
        # --- STAGE 2 ---
        x_s2 = self.s2_dssl1(x_s1, cond_feats)
        x_s2 = self.s2_ada1(x_s2, t_emb)
        
        x_in_s2 = torch.cat([feat_s1, x_s1], dim=1) 
        x_s2_res = self.s2_ConvNeXt(x_in_s2)
        feat_s2 = self.s2_proj(self.s2_ada2(x_s2_res, t_emb))

        # --- STAGE 3 ---
        x_s3 = self.s3_dssl1(x_s2,cond_feats)
        x_s3 = self.s3_ada1(x_s3, t_emb)
        
        x_in_s3 = torch.cat([feat_s2, x_s2], dim=1) 
        x_s3_res = self.s3_ConvNeXt(x_in_s3)
        feat_s3 = self.s3_proj(self.s3_ada2(x_s3_res, t_emb))

        out = self.final_gated_fusion(feat_s1, feat_s2, feat_s3)
        return self.final_conv(out)
        
    def forward(self, x, timesteps, condition_img):
        cond_feats = self.encode_condition(condition_img)
        return self.forward_with_cond(x, timesteps, cond_feats)
        
# ==========================================
# 4. TRAINING & UTILS
# ==========================================
def evaluate_vae_miou(vae, dataloader):
    print("\n" + "="*40)
    print(">>> CHECKING VAE RECONSTRUCTION QUALITY (mIoU)")
    print("="*40)
    vae.eval()
    miou_metric = MulticlassJaccardIndex(num_classes=NUM_CLASSES).to(DEVICE)
    with torch.no_grad():
        for _, masks in tqdm(dataloader, desc="Testing VAE"):
            masks = masks.to(DEVICE)
            target = torch.argmax(masks, dim=1)
            recon, _, _ = vae(masks)
            preds = torch.argmax(recon, dim=1)
            miou_metric.update(preds, target)
    final_miou = miou_metric.compute().item()
    print(f"✅ VAE Reconstruction mIoU: {final_miou:.4f}")
    if final_miou < 0.80: print("⚠️ WARNING: VAE mIoU is low.")
    elif final_miou > 0.95: print("🔥 EXCELLENT: VAE is perfect.")
    print("="*40 + "\n")
    return final_miou

def run_stage_1_vae():
    print("\n" + "="*40 + "\n>>> STAGE 1: VAE TRAINING\n" + "="*40)
    vae = SegmentationVAE(num_classes=NUM_CLASSES, latent_dim=VAE_LATENT_CHANNELS).to(DEVICE)
    
    checkpoint_path = None
    if os.path.exists(VAE_SAVE_PATH): checkpoint_path = VAE_SAVE_PATH
    elif os.path.exists(VAE_INPUT_PATH): checkpoint_path = VAE_INPUT_PATH
    
    if checkpoint_path:
        try:
            vae.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
            print("✅ VAE loaded."); return vae
        except: print("🔄 Load failed. Training new VAE...")
            
    if torch.cuda.device_count() > 1: vae = nn.DataParallel(vae)
    
    optimizer = torch.optim.AdamW(vae.parameters(), lr=1e-4)
    ce_loss = nn.CrossEntropyLoss()
    best_loss = float('inf')
    
    for epoch in range(30): 
        vae.train()
        total_loss = 0
        pbar = tqdm(train_dataloader, desc=f"VAE Ep {epoch+1}")
        for _, masks in pbar:
            masks = masks.to(DEVICE)
            target = torch.argmax(masks, dim=1)
            recon, mu, logvar = vae(masks)
            loss = ce_loss(recon, target) + 0.0001 * (-0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / (masks.shape[0]*masks.shape[2]*masks.shape[3]))
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({'Loss': f'{loss.item():.4f}'})
        
        avg_loss = total_loss / len(train_dataloader)
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(vae.module.state_dict() if hasattr(vae, 'module') else vae.state_dict(), VAE_BEST_PATH)
    
    return vae

def calculate_vae_scale_factor(vae, scale_factor=None):
    if scale_factor: return scale_factor
    print(">>> Calculating Scale Factor...")
    vae.eval()
    latents = []
    encode_fn = vae.module.encode if isinstance(vae, nn.DataParallel) else vae.encode
    with torch.no_grad():
        for _, masks in tqdm(train_dataloader):
            mu, _ = encode_fn(masks.to(DEVICE))
            latents.append(mu.cpu())
    std = torch.cat(latents, dim=0).std().item()
    print(f"✅ Scale Factor: {1.0/std:.4f}")
    return 1.0 / std

# ==========================================
# 5. STAGE 2: FLOW TRAINING (CONCAT FIXED)
# ==========================================
def run_stage_2_flow(vae, scale):
    print("\n" + "="*50)
    print(">>> STAGE 2: RECTIFIED FLOW (CONCATENATION ARCHITECTURE)")
    print("="*50)

    vae.eval()
    for p in vae.parameters(): p.requires_grad = False
    
    unet = myModel(
        latent_channels=VAE_LATENT_CHANNELS,
        cond_channels=COND_CHANNELS,
        base_channels=COND_LATENT_CHANNELS,
        out_channels=VAE_LATENT_CHANNELS
    ).to(DEVICE)

    if os.path.exists(FLOW_CHECKPOINT):
        print("✅ Loading Flow checkpoint")
        unet.load_state_dict(torch.load(FLOW_CHECKPOINT, map_location=DEVICE))

    if torch.cuda.device_count() > 1: unet = nn.DataParallel(unet)
    real_unet = unet.module if isinstance(unet, nn.DataParallel) else unet

    optimizer = torch.optim.AdamW(real_unet.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    mse_loss = nn.MSELoss()
    best_iou = 0.0

    vae_encode = vae.module.encode if isinstance(vae, nn.DataParallel) else vae.encode
    vae_reparam = vae.module.reparameterize if isinstance(vae, nn.DataParallel) else vae.reparameterize

    for epoch in range(20):
        unet.train()
        total_loss = 0.0 
        pbar = tqdm(train_dataloader, desc=f"Flow Epoch {epoch+1}")

        for i, (images, masks) in enumerate(pbar):
            images = images.to(DEVICE); masks = masks.to(DEVICE)
            
            # --- KHÔNG CẦN ENCODE THỦ CÔNG NỮA ---
            # Để unet tự lo việc này bên trong hàm forward()
            # Điều này giúp DataParallel chia tải đều cho các GPU

            # 1. Prepare Data & Noise
            with torch.no_grad():
                mu, logvar = vae_encode(masks)
                z_data = vae_reparam(mu, logvar) * scale 
            z_noise = torch.randn_like(z_data)
            
            # 2. Time
            t = torch.rand(z_data.shape[0], device=DEVICE)
            t_b = t.view(-1, 1, 1, 1)
            
            # 3. Interpolation
            zt = t_b * z_data + (1 - t_b) * z_noise
            target_v = z_data - z_noise 

            # 4. Forward: TRUYỀN THẲNG IMAGES
            # Class myModel mới sẽ tự gọi encode_condition bên trong
            pred_v = unet(zt, t, images) 

            loss = mse_loss(pred_v, target_v)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0) 
            optimizer.step()

            total_loss += loss.item()
            pbar.set_postfix({"MSE": f"{total_loss/(i+1):.4f}"})

        current_iou = validate_flow(vae, unet, scale, epoch, num_steps=1)
        
        print(f" LR: {optimizer.param_groups[0]['lr']:.2e}")
        scheduler.step(current_iou) 

        if current_iou > best_iou:
            best_iou = current_iou
            torch.save(real_unet.state_dict(), FLOW_BEST_PATH)
            print(f"🔥 New Best: {best_iou:.4f}")
        torch.save(real_unet.state_dict(), FLOW_CHECKPOINT)


@torch.no_grad()
def flow_inference_nstep(vae, unet, images, scale, num_steps=10):
    unet.eval()
    vae.eval()

    B = images.shape[0]
    dt = 1.0 / num_steps
    
    # Lấy model gốc (bỏ vỏ bọc DataParallel nếu có) để truy cập hàm con
    real_unet = unet.module if isinstance(unet, nn.DataParallel) else unet

    # 1. Encode Condition (Chạy 1 lần duy nhất)
    # Lưu ý: encode_condition đã có .contiguous() nhờ bước fix ở class myModel
    cond_feats = real_unet.encode_condition(images)

    # Khởi tạo latent ngẫu nhiên
    z = torch.randn(B, VAE_LATENT_CHANNELS, 32, 32, device=images.device)

    # Vòng lặp khử nhiễu
    for i in range(num_steps):
        t = torch.full((B,), i * dt, device=images.device)
        
        # 2. Gọi hàm FORWARD_WITH_COND (để dùng feature đã cache)
        # Thay vì gọi unet(z, t, images) -> sẽ bị encode lại rất chậm
        v = real_unet.forward_with_cond(z, t, cond_feats)
        
        z = z + v * dt

    logits = vae.decode(z / scale)
    return logits


def validate_flow(vae, unet, scale, epoch, num_steps=10):
    iou_metric = MulticlassJaccardIndex(num_classes=NUM_CLASSES).to(DEVICE)
    real_vae = vae.module if isinstance(vae, nn.DataParallel) else vae
    print(f">>> Val (Euler steps = {num_steps})")
    pbar = tqdm(val_dataloader, desc="Val")

    for images, labels in pbar:
        images = images.to(DEVICE)
        gt = torch.argmax(labels, dim=1).to(DEVICE)

        with torch.no_grad():
            logits = flow_inference_nstep(
                real_vae,
                unet,
                images,
                scale,
                num_steps=num_steps
            )
        preds = logits.argmax(dim=1)
        iou_metric.update(preds, gt)
        mean_iou = iou_metric.compute().item()
        pbar.set_postfix({
            "mIoU": f"{mean_iou:.4f}"
        })
    miou = iou_metric.compute().item()
    print(f"Epoch {epoch+1} mIoU: {miou:.4f}")
    return miou



# ==========================================
# UTILS: MODEL STATISTICS (THOP)
# ==========================================
def print_model_stats(model, inputs, name="Model"):
    try:
        from thop import profile, clever_format
        model.eval()
        # Chuyển inputs sang device
        inputs = tuple(i.to(DEVICE) for i in inputs)
        
        # Tính toán
        macs, params = profile(model, inputs=inputs, verbose=False)
        macs_fmt, params_fmt = clever_format([macs, params], "%.3f")
        
        print(f"📊 {name}:")
        print(f"   - Params: {params_fmt}")
        print(f"   - GFLOPs: {macs_fmt}")
    except ImportError:
        print("⚠️ Library 'thop' not installed. Skipped profiling. (pip install thop)")
    except Exception as e:
        print(f"⚠️ Could not profile {name}: {e}")

def print_all_stats():
    print("\n" + "="*40)
    print(">>> MODEL ANALYSIS (CONCAT ARCHITECTURE)")
    print("="*40)
    
    # 1. VAE (Giữ nguyên)
    vae = SegmentationVAE(num_classes=NUM_CLASSES, latent_dim=VAE_LATENT_CHANNELS).to(DEVICE)
    dummy_mask = torch.randn(1, NUM_CLASSES, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
    print_model_stats(vae, (dummy_mask,), "Segmentation VAE")

    # 2. Flow Model
    unet = myModel(
        latent_channels=VAE_LATENT_CHANNELS,
        cond_channels=COND_CHANNELS,
        base_channels=COND_LATENT_CHANNELS,
        out_channels=VAE_LATENT_CHANNELS
    ).to(DEVICE)
    
    # Dummy Inputs
    dummy_z = torch.randn(1, VAE_LATENT_CHANNELS, 32, 32).to(DEVICE)
    dummy_t = torch.tensor([0.5]).to(DEVICE)
    dummy_img = torch.randn(1, COND_CHANNELS, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
    
    # A. Profile Encoder Riêng
    print("--- Sub-module: Condition Encoder ---")
    print_model_stats(unet.cond_enc, (dummy_img,), "ResNet34 Encoder")
    
    # B. Profile Full Flow Network (Core Only)
    # --- FIX LỖI Ở ĐÂY ---
    print("--- Full Flow Network (Core / Denoising Only) ---")
    
    # 1. Tạo features giả lập (đã encode xong)
    with torch.no_grad():
        dummy_cond_feats = unet.encode_condition(dummy_img)

    # 2. Tạo một Wrapper Class tạm thời để THOP hiểu được
    class CoreWrapper(nn.Module):
        def __init__(self, origin_model):
            super().__init__()
            self.model = origin_model
        def forward(self, z, t, cond_feats):
            # Gọi đúng hàm forward_with_cond
            return self.model.forward_with_cond(z, t, cond_feats)

    # 3. Profile trên Wrapper này
    unet_core_wrapper = CoreWrapper(unet).to(DEVICE)
    print_model_stats(unet_core_wrapper, (dummy_z, dummy_t, dummy_cond_feats), "Flow UNet Core")
    
    # Dọn dẹp
    del vae, unet, unet_core_wrapper, dummy_mask, dummy_z, dummy_t, dummy_img, dummy_cond_feats
    torch.cuda.empty_cache()
    print("="*40 + "\n")
    
if __name__ == "__main__":
    print_all_stats()
    vae_model = run_stage_1_vae()
    # evaluate_vae_miou(vae_model, val_dataloader)
    scale = calculate_vae_scale_factor(vae_model, scale_factor=1.0) #1.3238
    run_stage_2_flow(vae_model, scale)

In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import copy
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from torchmetrics.classification import MulticlassJaccardIndex
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision.models as models

# ==========================================
# 1. CẤU HÌNH & PATHS
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE = 256
BATCH_SIZE = 32
NUM_CLASSES = 4            
COND_CHANNELS = 3          

# Model Architecture Config
VAE_LATENT_CHANNELS = 4    
COND_LATENT_CHANNELS = 128 

# Checkpoint Paths (CẦN CHECK FILE TỒN TẠI)
VAE_BEST_PATH = "/kaggle/input/vae07/pytorch/default/1/custom_seg_vae_best (1).pth"       # Path model VAE đã train ở Stage 1
TEACHER_CHECKPOINT = "best_latent_flow_concat_opt.pth" # Path model Flow đã train ở Stage 2
REFLOW_SAVE_PATH = "reflow_model.pth"
REFLOW_BEST_PATH = "best_reflow_model.pth"

# ==========================================
# 2. DATASET & TRANSFORMS
# ==========================================
class SemanticSegmentationDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform
        self.image_paths = sorted([os.path.join(image_dir, img) for img in os.listdir(image_dir)])
        self.label_paths = sorted([os.path.join(label_dir, lbl) for lbl in os.listdir(label_dir)])
        self.class_colors = {(255, 255, 255): 0, (160, 160, 160): 1, (80, 80, 80): 2, (0, 0, 0): 3}
     
    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        image = cv2.imread(self.image_paths[idx]); image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = cv2.imread(self.label_paths[idx]); label = cv2.cvtColor(label, cv2.COLOR_BGR2RGB)
        
        label_mask = np.zeros(label.shape[:2], dtype=np.uint8)
        for rgb, idx in self.class_colors.items():
            label_mask[np.all(label == rgb, axis=-1)] = idx
            
        if self.transform:
            image = self.transform(image)
            label_mask = torch.from_numpy(label_mask).long()
            label_onehot = F.one_hot(label_mask, num_classes=NUM_CLASSES).permute(2, 0, 1).float()
            
        return image, label_onehot

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# [LƯU Ý] CẬP NHẬT ĐƯỜNG DẪN CỦA BẠN
train_dataset = SemanticSegmentationDataset(
    image_dir='/kaggle/input/5g-lte-nr-j03/J03_spectrumm/train/data', 
    label_dir='/kaggle/input/5g-lte-nr-j03/J03_spectrumm/train/label', 
    transform=train_transform
)
val_dataset = SemanticSegmentationDataset(
    image_dir='/kaggle/input/5g-lte-nr-j03/J03_spectrumm/test/data', 
    label_dir='/kaggle/input/5g-lte-nr-j03/J03_spectrumm/test/label', 
    transform=train_transform
)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

# ==========================================
# 3. MODEL CLASSES (VAE + FLOW)
# ==========================================

class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

class AdaBatchNorm(nn.Module):
    def __init__(self, channels, time_dim):
        super().__init__()
        self.bn = nn.BatchNorm2d(channels, affine=False)
        self.emb = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_dim, channels * 2)
        )

    def forward(self, x, t_emb):
        scale_shift = self.emb(t_emb)[:, :, None, None]
        scale, shift = scale_shift.chunk(2, dim=1)
        x = self.bn(x)
        return x * (1 + scale) + shift


class ConvNeXtBlockBN(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, 7, padding=3, groups=dim)
        self.bn = nn.BatchNorm2d(dim)
        self.pwconv1 = nn.Conv2d(dim, 4 * dim, 1)
        self.act = nn.SiLU()
        self.pwconv2 = nn.Conv2d(4 * dim, dim, 1)
        self.bn2 = nn.BatchNorm2d(4 * dim)
    def forward(self, x):
        identity = x
        x = self.dwconv(x)
        x = self.bn(x)
        x = self.pwconv1(x)
        x = self.bn2(x)
        x = self.pwconv2(x)
        x = self.act(x)
        return x + identity


# ==============================================================================
# 2. VAE BLOCKS (GIỮ NGUYÊN LOGIC, CHỈ FIX LỖI RESUPBLOCK)
# ==============================================================================
# --- BLOCKS CHO VAE ---
class ResDownBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, 2, 1), nn.BatchNorm2d(out_c), nn.GELU(),
            nn.Conv2d(out_c, out_c, 3, 1, 1), nn.BatchNorm2d(out_c), nn.GELU()
        )
        self.skip = nn.Sequential(nn.Conv2d(in_c, out_c, 1, 2), nn.BatchNorm2d(out_c))
    def forward(self, x): return self.conv(x) + self.skip(x)

class ResUpBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_c, in_c, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, 1, 1), nn.BatchNorm2d(out_c), nn.GELU(),
            nn.Conv2d(out_c, out_c, 3, 1, 1), nn.BatchNorm2d(out_c), nn.GELU()
        )
        self.skip = nn.Conv2d(in_c, out_c, 1)
    def forward(self, x): 
        x_up = self.up(x)
        return self.conv(x_up) + self.skip(x_up)

class SegmentationVAE(nn.Module):
    def __init__(self, num_classes=4, latent_dim=4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(num_classes, 32, 3, 1, 1), nn.GELU(),
            ResDownBlock(32, 32), ResDownBlock(32, 48), ResDownBlock(48, 96),
        )
        self.fc_mu = nn.Conv2d(96, latent_dim, 1)
        self.fc_logvar = nn.Conv2d(96, latent_dim, 1)
        
        self.decoder_input = nn.Conv2d(latent_dim, 96, 1)
        self.decoder = nn.Sequential(
            ResUpBlock(96, 48), ResUpBlock(48, 32), ResUpBlock(32, 32),
            nn.Conv2d(32, num_classes, 1)
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(self.decoder_input(z))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar
# ==============================================================================
# 3. ATTENTION & DSSL BLOCKS (OPTIMIZED)
# ==============================================================================
class GatedFeatureAggregation(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate_proj = nn.Sequential(
            nn.Conv2d(dim * 3, dim, 3,1,1, bias=False),
            nn.BatchNorm2d(dim),
            nn.SiLU(), 
            nn.Conv2d(dim, dim * 3, 3,1,1, bias=False),
            nn.BatchNorm2d(dim * 3),
            nn.Sigmoid()
        )
        # self.out_proj = nn.Sequential(
        #     nn.Conv2d(dim, dim, 3, 1, 1, groups=dim, bias=False),
        #     nn.GroupNorm(8, dim),
        #     nn.SiLU() 
        # )
    def forward(self, f1, f2, f3):
        cat_feats = torch.cat([f1, f2, f3], dim=1)
        weights = self.gate_proj(cat_feats)
        w1, w2, w3 = weights.chunk(3, dim=1)
        
        aggregated = (f1 * w1) + (f2 * w2) + (f3 * w3)
        return aggregated

def haar_wavelet_decompose(x):
    B, C, H, W = x.shape
    if H % 2 != 0 or W % 2 != 0: x = F.pad(x, (0, W % 2, 0, H % 2), mode='reflect')
    x00 = x[:, :, 0::2, 0::2]; x01 = x[:, :, 0::2, 1::2]
    x10 = x[:, :, 1::2, 0::2]; x11 = x[:, :, 1::2, 1::2]
    LL = (x00 + x01 + x10 + x11) / 4
    LH = (x00 + x01 - x10 - x11) / 4
    HL = (x00 - x01 + x10 - x11) / 4
    HH = (x00 - x01 - x10 + x11) / 4
    return LL, torch.cat([LH, HL, HH], dim=1)

class DSSL(nn.Module):
    """
    Dilated Scale-Sensitive Layer (Optimized):
    Xử lý trực tiếp trên High/Low freq maps, bỏ qua bước interpolate dư thừa.
    """
    def __init__(self, channels=96, kernel_size=3, dilations=[2, 4, 8]):
        super().__init__()
        self.num_dilated_paths = len(dilations)
        self.conv_paths = nn.ModuleList()
        for d in dilations:
            self.conv_paths.append(nn.Sequential(
                nn.Conv2d(channels, channels, kernel_size, padding=d, dilation=d, groups=channels, bias=False),
                nn.BatchNorm2d(channels),
                nn.SiLU(), 
                nn.Conv2d(channels, channels, 1, bias=False)
            ))
        # Low freq processor
        self.process_LL = nn.Sequential(
            nn.Conv2d(channels, channels, 3, 1, 1, bias=False), 
            nn.BatchNorm2d(channels), nn.SiLU() 
        )
        # High freq processor
        self.process_High = nn.Sequential(
            nn.Conv2d(channels * 3, channels, 3, 1, 1, bias=False), 
            nn.BatchNorm2d(channels), nn.SiLU() 
        )
        self.global_pool = nn.AdaptiveAvgPool2d(1) 
        
        self.attention_mlp = nn.Sequential(
            nn.Conv2d(channels * 2, channels // 2, 1), 
            nn.SiLU(), 
            nn.Conv2d(channels // 2, self.num_dilated_paths * channels, 1) 
        )
        self.sigmoid = nn.Sigmoid()
        self.final_proj = nn.Conv2d(channels, channels, 1)

    def forward(self, x, cond):
        B, C, H, W = x.shape
        dilated_outs = torch.stack([conv(x) for conv in self.conv_paths], dim=1)
        
        # Haar Decompose -> Ra size H/2, W/2
        LL, High_Freqs = haar_wavelet_decompose(cond)
        
        feat_LL = self.process_LL(LL)           
        feat_High = self.process_High(High_Freqs) 
        
        # Pool trực tiếp từ size nhỏ
        pool_LL = self.global_pool(feat_LL)     
        pool_High = self.global_pool(feat_High) 
        
        ctx = torch.cat([pool_LL, pool_High], dim=1) 
        gates = self.attention_mlp(ctx)              
        gates = gates.view(B, self.num_dilated_paths, C, 1, 1) 
        gates = self.sigmoid(gates)
        
        out = torch.sum(dilated_outs * gates, dim=1) 
        return self.final_proj(out) + x


# ==========================================
# 2. CLASS ENCODER ĐÃ FIX LỖI
# ==========================================
class ResNetConditionEncoder(nn.Module):
    def __init__(self, in_channels=3, out_channels=96, base_dim=96):
        super().__init__()
        print(f"⚡ Initializing SpectraNeXtEncoder (Modern ConvNeXt-style) | Out: {out_channels}")

        # 1. Stem: Nén ảnh nhanh (256x256 -> 32x32) qua 3 bước stride=2
        self.stem = nn.Sequential(
            # Bước 1: 256 -> 128
            nn.Conv2d(in_channels, base_dim//2, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_dim//2),
            nn.SiLU(),
            nn.Conv2d(base_dim//2, base_dim//2, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(base_dim//2),
            nn.SiLU(),

            # Bước 2: 128 -> 64
            nn.Conv2d(base_dim//2, base_dim, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_dim),
            nn.SiLU(),
            nn.Conv2d(base_dim, base_dim, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(base_dim),
            nn.SiLU(),

            # Bước 3: 64 -> 32
            nn.Conv2d(base_dim, int(base_dim*2), kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(int(base_dim*2)),
            nn.SiLU(),
            nn.Conv2d(int(base_dim*2), int(base_dim*2), kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(int(base_dim*2)),
            nn.SiLU(),
        )
        
        self.final_proj = nn.Sequential(
            nn.Conv2d(int(base_dim*2), out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU()
        )

    def forward(self, x):
        x = self.stem(x)
        return self.final_proj(x)        
        
class myModel(nn.Module):
    def __init__(self, latent_channels=4, cond_channels=3, cond_inner_dim=96, out_channels=4, base_channels=96):
        super().__init__()
        print(f"🚀 Initializing Optimized UNet (Concat Only) | Base: {base_channels}")
        
        self.base_channels = base_channels
        self.cond_inner_dim = cond_inner_dim
        time_dim = base_channels * 4
        
        # 1. Condition Encoder (ResNet - Đã fix logic như bài trước)
        self.cond_enc = ResNetConditionEncoder(out_channels=cond_inner_dim)
        
        # 2. Time Embedding
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(base_channels),
            nn.Linear(base_channels, time_dim), nn.SiLU(), nn.Linear(time_dim, time_dim),
        )
        
        # 3. Stem
        self.stem = nn.Sequential(
            nn.Conv2d(latent_channels, base_channels, 3, 1, 1),
            ConvNeXtBlockBN(base_channels), ConvNeXtBlockBN(base_channels)
        )
        self.ada_stem = AdaBatchNorm(base_channels, time_dim)
        
        # STAGE 1
        self.s1_dssl1 = DSSL(base_channels, dilations=[2, 4, 8])
        self.s1_ada1  = AdaBatchNorm(base_channels, time_dim)
        self.s1_ConvNeXt = nn.Sequential(
            ConvNeXtBlockBN(base_channels + cond_inner_dim), ConvNeXtBlockBN(base_channels + cond_inner_dim)
        )
        self.s1_ada2  = AdaBatchNorm(base_channels + cond_inner_dim, time_dim)
        self.s1_proj  = nn.Conv2d(base_channels + cond_inner_dim, base_channels, 3, 1, 1)
        
        # STAGE 2
        self.s2_dssl1 = DSSL(base_channels, dilations=[2, 4, 8])
        self.s2_ada1  = AdaBatchNorm(base_channels, time_dim)
        self.s2_ConvNeXt = nn.Sequential(
            ConvNeXtBlockBN(base_channels + cond_inner_dim), ConvNeXtBlockBN(base_channels + cond_inner_dim)
        )
        self.s2_ada2  = AdaBatchNorm(base_channels + cond_inner_dim, time_dim)
        self.s2_proj  = nn.Conv2d(base_channels + cond_inner_dim, base_channels, 3, 1, 1) 
        
        # STAGE 3 
        self.s3_dssl1 = DSSL(base_channels, dilations=[2, 4, 8])
        self.s3_ada1  = AdaBatchNorm(base_channels, time_dim)
        self.s3_ConvNeXt = nn.Sequential(
            ConvNeXtBlockBN(base_channels + cond_inner_dim), ConvNeXtBlockBN(base_channels + cond_inner_dim)
        )
        self.s3_ada2  = AdaBatchNorm(base_channels + cond_inner_dim, time_dim)
        self.s3_proj = nn.Conv2d(base_channels + cond_inner_dim, base_channels, 3, 1, 1)

        # FINAL
        self.final_gated_fusion = GatedFeatureAggregation(base_channels)
        self.final_conv = nn.Conv2d(base_channels, out_channels, 3, 1, 1)

    def encode_condition(self, condition_img):
        return self.cond_enc(condition_img)

    def forward_with_cond(self, x, timesteps, cond_feats):
        t_emb = self.time_mlp(timesteps)
        x_stem = self.stem(x)
        x_stem = self.ada_stem(x_stem, t_emb)
        
        # stage 1
        x_s1 = self.s1_dssl1(x_stem,cond_feats)
        x_s1 = self.s1_ada1(x_s1, t_emb)
        
        x_in_s1 = torch.cat([x_stem, cond_feats], dim=1)  
        x_s1_res = self.s1_ConvNeXt(x_in_s1)
        feat_s1 = self.s1_proj(self.s1_ada2(x_s1_res, t_emb))
        
        # --- STAGE 2 ---
        x_s2 = self.s2_dssl1(x_s1, cond_feats)
        x_s2 = self.s2_ada1(x_s2, t_emb)
        
        x_in_s2 = torch.cat([feat_s1, x_s1], dim=1) 
        x_s2_res = self.s2_ConvNeXt(x_in_s2)
        feat_s2 = self.s2_proj(self.s2_ada2(x_s2_res, t_emb))

        # --- STAGE 3 ---
        x_s3 = self.s3_dssl1(x_s2,cond_feats)
        x_s3 = self.s3_ada1(x_s3, t_emb)
        
        x_in_s3 = torch.cat([feat_s2, x_s2], dim=1) 
        x_s3_res = self.s3_ConvNeXt(x_in_s3)
        feat_s3 = self.s3_proj(self.s3_ada2(x_s3_res, t_emb))

        out = self.final_gated_fusion(feat_s1, feat_s2, feat_s3)
        return self.final_conv(out)
        
    def forward(self, x, timesteps, condition_img):
        cond_feats = self.encode_condition(condition_img)
        return self.forward_with_cond(x, timesteps, cond_feats)


# ==========================================
# 4. REFLOW HELPERS & LOGIC
# ==========================================
class ReflowLinkedDataset(Dataset):
    def __init__(self, reflow_data_list, original_dataset):
        self.reflow_data = reflow_data_list
        self.original_dataset = original_dataset
    def __len__(self): return len(self.reflow_data)
    def __getitem__(self, idx):
        item = self.reflow_data[idx]
        return item['z0'], item['z1'], self.original_dataset[item['original_idx']][0]

@torch.no_grad()
def get_vae_and_scale():
    print(f"🔄 Loading VAE from {VAE_BEST_PATH}...")
    vae = SegmentationVAE(NUM_CLASSES, VAE_LATENT_CHANNELS).to(DEVICE)
    if os.path.exists(VAE_BEST_PATH):
        vae.load_state_dict(torch.load(VAE_BEST_PATH, map_location=DEVICE))
    else:
        raise FileNotFoundError(f"❌ Không tìm thấy file {VAE_BEST_PATH}! Hãy chạy Stage 1 trước.")
    vae.eval()
    
    print("⚖️ Calculating Scale Factor (Sample 1 batch)...")
    # Tính nhanh scale factor dựa trên 1 vài batch đầu để tiết kiệm thời gian
    # Nếu muốn chính xác tuyệt đối, bỏ break và chạy hết loop
    latents = []
    limit_batches = 10 
    for i, (_, masks) in enumerate(train_dataloader):
        mu, _ = vae.encode(masks.to(DEVICE))
        latents.append(mu.cpu())
        if i >= limit_batches: break
    
    std = torch.cat(latents, dim=0).std().item()
    std=1.0 # bật tắt scale
    scale = 1.0 / std
    print(f"✅ Scale Factor: {scale:.4f}")
    return vae, scale

@torch.no_grad()
def validate_reflow_1step(vae, unet, scale_factor, epoch):
    print(f"\n>>> Validating Reflow 1-Step (Epoch {epoch+1})...")
    unet.eval(); vae.eval()
    iou_metric = MulticlassJaccardIndex(num_classes=NUM_CLASSES).to(DEVICE)
    net = unet.module if isinstance(unet, nn.DataParallel) else unet

    for images, labels in tqdm(val_dataloader, desc="Val Reflow", leave=False):
        images = images.to(DEVICE); gt = torch.argmax(labels, dim=1).to(DEVICE)
        B = images.shape[0]
        
        # 1-Step Inference Optimized
        cond_feats = net.encode_condition(images)
        z0 = torch.randn(B, VAE_LATENT_CHANNELS, 32, 32, device=DEVICE)
        t = torch.zeros(B, device=DEVICE)
        
        v = net.forward_with_cond(z0, t, cond_feats)
        z_pred = z0 + v 
        
        logits = vae.decode(z_pred / scale_factor)
        iou_metric.update(logits.argmax(dim=1), gt)

    final_iou = iou_metric.compute().item()
    print(f"🔥 Reflow 1-Step mIoU: {final_iou:.4f}")
    return final_iou

def run_stage_3_reflow(vae, scale_factor):
    print("\n" + "="*50 + "\n>>> STAGE 3: RECTIFIED FLOW (REFLOW)\n" + "="*50)

    # 1. LOAD TEACHER
    print(f"🎓 Loading Teacher from {TEACHER_CHECKPOINT}...")
    teacher = myModel(VAE_LATENT_CHANNELS, COND_CHANNELS, COND_LATENT_CHANNELS, VAE_LATENT_CHANNELS).to(DEVICE)
    if not os.path.exists(TEACHER_CHECKPOINT):
        raise FileNotFoundError("❌ Không tìm thấy Teacher checkpoint! Hãy chạy Stage 2 trước.")
        
    teacher.load_state_dict(torch.load(TEACHER_CHECKPOINT, map_location=DEVICE))
    teacher.eval()
    if torch.cuda.device_count() > 1: teacher = nn.DataParallel(teacher)

    # 2. GENERATE DATA
    print("⏳ Generating (z0, z1) pairs (50 steps)...")
    reflow_data = []
    gen_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    steps = 50; dt = 1.0 / steps
    net_teacher = teacher.module if isinstance(teacher, nn.DataParallel) else teacher

    with torch.no_grad():
        global_idx = 0
        for images, _ in tqdm(gen_loader, desc="Gen Trajectories"):
            images = images.to(DEVICE)
            B = images.shape[0]
            cond_feats = net_teacher.encode_condition(images)
            z0 = torch.randn(B, VAE_LATENT_CHANNELS, 32, 32, device=DEVICE)
            z_curr = z0.clone()
            
            for k in range(steps):
                t = torch.full((B,), k / steps, device=DEVICE)
                v = net_teacher.forward_with_cond(z_curr, t, cond_feats)
                z_curr = z_curr + v * dt
            
            z0_cpu, z1_cpu = z0.cpu(), z_curr.cpu()
            for i in range(B):
                reflow_data.append({'z0': z0_cpu[i], 'z1': z1_cpu[i], 'original_idx': global_idx})
                global_idx += 1

    del teacher, net_teacher, cond_feats, z0, z_curr; torch.cuda.empty_cache()
    print(f"✅ Generated {len(reflow_data)} pairs.")

    # 3. TRAIN STUDENT
    print("🚀 Initializing Student Model...")
    student = myModel(VAE_LATENT_CHANNELS, COND_CHANNELS, COND_LATENT_CHANNELS, VAE_LATENT_CHANNELS).to(DEVICE)
    try:
        student.load_state_dict(torch.load(TEACHER_CHECKPOINT, map_location=DEVICE))
        print("💡 Student initialized with Teacher weights.")
    except: print("⚠️ Training Student from scratch.")

    if torch.cuda.device_count() > 1: student = nn.DataParallel(student)
    real_student = student.module if isinstance(student, nn.DataParallel) else student

    train_loader = DataLoader(ReflowLinkedDataset(reflow_data, train_dataset), batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    optimizer = torch.optim.AdamW(real_student.parameters(), lr=1e-5, weight_decay=1e-5)
    scheduler = CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-6)
    mse_loss = nn.MSELoss()
    best_iou = 0.0

    print("⚔️  Start Reflow Training...")
    for epoch in range(30):
        student.train()
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"Reflow Ep {epoch+1}")
        
        for z0, z1, cond_imgs in pbar:
            z0, z1, cond_imgs = z0.to(DEVICE), z1.to(DEVICE), cond_imgs.to(DEVICE)
            target_v = z1 - z0
            t = torch.rand(z0.shape[0], device=DEVICE)
            z_t = t.view(-1,1,1,1) * z1 + (1 - t.view(-1,1,1,1)) * z0
            
            pred_v = student(z_t, t, cond_imgs)
            loss = mse_loss(pred_v, target_v)
            
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({'MSE': f"{loss.item():.5f}"})
        
        scheduler.step()
        current_iou = validate_reflow_1step(vae, student, scale_factor, epoch)
        if current_iou > best_iou:
            best_iou = current_iou
            torch.save(real_student.state_dict(), REFLOW_BEST_PATH)
            print(f"💎 New Best Reflow Model: {best_iou:.4f}")
        torch.save(real_student.state_dict(), REFLOW_SAVE_PATH)

# ==========================================
# 5. EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    # 1. Load VAE Context (Mới thêm vào)
    vae_model, scale_factor = get_vae_and_scale()
    
    # 2. Run Reflow
    run_stage_3_reflow(vae_model, scale_factor)